# Qwen3 Advanced GRPO Training — Google Colab

**Prerequisites:** Runtime > Change runtime type > **T4 GPU** (free) or A100 (Colab Pro)

| Model | VRAM needed | Fits on |
|-------|------------|--------|
| Qwen3-1.7B | ~4 GB | T4 (free) |
| Qwen3-4B   | ~6 GB | T4 (free) |
| Qwen3-8B   | ~10 GB | T4 (free, tight) |
| Qwen3-14B  | ~16 GB | A100 (Pro) |


In [ ]:
# Step 0: Verify GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — change runtime type to GPU')

In [ ]:
# Step 1: Install Unsloth + dependencies
# Colab already has PyTorch with CUDA — Unsloth installs on top
!pip install -q "unsloth[huggingface]" "trl>=0.18.2,<=0.24.0" wandb

In [ ]:
# Step 2: HuggingFace login (paste your token from https://hf.co/settings/tokens)
from huggingface_hub import login
login()  # Opens a widget to enter your token

In [ ]:
# Step 3: Configuration — edit these
MODEL_NAME          = "unsloth/Qwen3-4B"   # Use 4B for free T4; 8B if VRAM allows
MAX_SEQ_LENGTH      = 4096
LOAD_IN_4BIT        = True
LORA_RANK           = 16
OUTPUT_DIR          = "/content/qwen3-grpo-output"
NUM_GENERATIONS     = 4          # Rollouts per prompt (reduce if OOM)
MAX_COMPLETION_LEN  = 512
LEARNING_RATE       = 5e-6
MAX_STEPS           = 100        # Quick run; set -1 for full epoch
DATASET_NAME        = "openai/gsm8k"
REPORT_TO           = "none"     # Set "wandb" if you want W&B logging

In [ ]:
# Step 4: Load Qwen3 model via Unsloth
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = LOAD_IN_4BIT,
    dtype          = None,   # Auto-detect bfloat16/float16
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    lora_alpha     = LORA_RANK,
    lora_dropout   = 0.0,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Step 5: Multi-objective reward functions
import re
import math

_BOXED_RE    = re.compile(r"\\boxed\{(.+?)\}", re.DOTALL)
_THINKING_RE = re.compile(r"<think>(.*?)</think>", re.DOTALL)
_STEP_RE     = re.compile(r"(?:step\s*\d+|\d+\.\s|\n-\s)", re.IGNORECASE)

def format_reward(completions, **kwargs):
    """Reward <think> block + \\boxed{} answer format."""
    scores = []
    for comp in completions:
        score = 0.0
        thinking = _THINKING_RE.search(comp)
        if thinking:
            score += 0.5
            if len(thinking.group(1).split()) >= 10:
                score += 0.2
        if _BOXED_RE.search(comp):
            score += 0.3
        scores.append(score)
    return scores

def correctness_reward(completions, answers=None, **kwargs):
    """Numerical answer verification against ground truth."""
    if not answers:
        return [0.0] * len(completions)
    scores = []
    for comp, expected in zip(completions, answers):
        score = 0.0
        hits = _BOXED_RE.findall(comp)
        predicted = hits[-1].strip() if hits else None
        if predicted:
            try:
                if math.isclose(float(predicted.replace(",", "")),
                                float(str(expected).replace(",", "")), rel_tol=1e-4):
                    score = 1.0
                elif str(expected) in comp:
                    score = 0.3
            except ValueError:
                pass
        scores.append(score)
    return scores

def reasoning_reward(completions, **kwargs):
    """Reward structured step-by-step reasoning in <think>."""
    scores = []
    for comp in completions:
        m = _THINKING_RE.search(comp)
        thinking = m.group(1) if m else ""
        if not thinking:
            scores.append(0.0)
            continue
        score  = min(len(_STEP_RE.findall(thinking)) * 0.08, 0.4)
        score += min(thinking.count("=") * 0.03, 0.2)
        score += min(sum(w in thinking.lower()
                        for w in ["therefore", "because", "hence", "thus"]) * 0.05, 0.2)
        scores.append(round(min(score, 1.0), 4))
    return scores

def length_reward(completions, **kwargs):
    """Gaussian reward peaked at ~350 words."""
    return [round(math.exp(-0.5 * ((len(c.split()) - 350) / 200) ** 2), 4)
            for c in completions]

def composite_reward(completions, **kwargs):
    return [
        0.30 * f + 0.40 * c + 0.20 * r + 0.10 * ln
        for f, c, r, ln in zip(
            format_reward(completions, **kwargs),
            correctness_reward(completions, **kwargs),
            reasoning_reward(completions, **kwargs),
            length_reward(completions, **kwargs),
        )
    ]

print("Reward functions defined.")

In [ ]:
# Step 6: Load and format dataset
from datasets import load_dataset

SYSTEM_PROMPT = (
    "You are a rigorous reasoning assistant. "
    "Think step-by-step inside <think>...</think> before answering. "
    "Always put your final numerical answer inside \\boxed{}."
)

def build_prompt(example):
    question = example.get("question", "")
    answer   = example.get("answer", "")
    numeric  = answer.split("####")[-1].strip().replace(",", "") if "####" in str(answer) else ""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,   # Qwen3 thinking mode
    )
    return {"prompt": prompt_text, "answer": numeric}

raw_train = load_dataset(DATASET_NAME, "main", split="train")
raw_eval  = load_dataset(DATASET_NAME, "main", split="test").select(range(100))

train_ds = raw_train.map(build_prompt, remove_columns=raw_train.column_names)
eval_ds  = raw_eval.map(build_prompt,  remove_columns=raw_eval.column_names)

print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")

In [ ]:
# Step 7: Enable Unsloth GRPO optimisation patch
from unsloth.models.rl import PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)
print("Unsloth GRPO patch applied.")

In [ ]:
# Step 8: Configure and launch GRPO training
import torch
from trl import GRPOConfig, GRPOTrainer

use_bf16 = torch.cuda.is_bf16_supported()   # True on A100; False on T4
use_fp16 = not use_bf16
print(f"Mixed precision: {'bf16' if use_bf16 else 'fp16'}")

training_args = GRPOConfig(
    output_dir                  = OUTPUT_DIR,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.05,
    max_steps                   = MAX_STEPS,
    fp16                        = use_fp16,
    bf16                        = use_bf16,
    num_generations             = NUM_GENERATIONS,
    max_prompt_length           = 512,
    max_completion_length       = MAX_COMPLETION_LEN,
    temperature                 = 0.9,
    logging_steps               = 1,
    save_steps                  = 50,
    eval_steps                  = 50,
    evaluation_strategy         = "steps",
    report_to                   = REPORT_TO,
    remove_unused_columns       = False,
)

trainer = GRPOTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = eval_ds,
    reward_funcs  = [composite_reward],
    args          = training_args,
)

print("Starting GRPO training...")
trainer.train()

In [ ]:
# Step 9: Save the fine-tuned model
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to: {OUTPUT_DIR}")

In [ ]:
# Step 10 (optional): Push to HuggingFace Hub
# HUB_REPO = "your-username/qwen3-grpo-gsm8k"   # uncomment and set
# model.push_to_hub(HUB_REPO)
# tokenizer.push_to_hub(HUB_REPO)
# print(f"Pushed to: https://huggingface.co/{HUB_REPO}")

In [ ]:
# Step 11: Quick inference test
FastLanguageModel.for_inference(model)

test_question = "A store offers a 20% discount on a $150 item. If tax is 8%, what is the final price?"
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": test_question},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.7, do_sample=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=False))